# LangChain: Models, Prompts and Output Parsers

## Outline

本节课涉及三个 LangChain 核心概念：
* **Models（模型）**：如何统一封装、调用不同的语言模型
* **Prompts（提示词模板）**：如何把 prompt 结构和具体输入解耦，实现可复用
* **Output Parsers（输出解析器）**：如何让模型输出结构化、可被程序直接使用的数据（而不是只能人读的文本）

> 注：本 notebook 已针对 langchain 1.4.0 等新版本做了 import 路径和调用方式的修复，
> 并补充了中文注释；涉及真实网络请求（调用 OpenAI）的 cell，在没有真实 API Key 时会在联网阶段报错，这是预期行为。

In [ ]:
# 读取项目根目录下的 .env 文件，把里面的环境变量（如 OPENAI_API_KEY）加载到 os.environ 中
# find_dotenv() 会自动向上查找 .env 文件，load_dotenv() 负责真正加载
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']  # 给 openai 库设置全局 API Key（旧版 openai<1.0 的用法）

In [ ]:
# 课程录制于 2023 年，当时 gpt-3.5-turbo 会在 2024-06-12 之后废弃旧的 0301 版本，
# 所以用当前日期判断该用哪个模型名。现在（2026 年）早已过了这个日期，会自动选用 "gpt-3.5-turbo"。
# 这段逻辑本身没有 bug，只是单纯的日期判断，保留原样。
import datetime
current_data=datetime.datetime.now().date()
target_data=datetime.date(2024,6,12)
if current_data>target_data:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

## Chat API : OpenAI

先不用 LangChain，直接用 openai 官方库封装一个最简单的 `get_completion` 函数，
体会一下"手写调用"的样子，为后面对比 LangChain 的封装方式做铺垫。

In [ ]:
# 封装一个调用 OpenAI Chat 接口的小函数
# 【版本兼容性修复】原代码用的是 openai.ChatCompletion.create(...)，
# 这是 openai<1.0 时代的旧写法。当前装的是 openai>=1.0（这里是 3.10.0），
# openai.ChatCompletion 已被替换为一个"访问即报错"的占位对象，
# 调用 .create() 会直接抛出 APIRemovedInV1 异常，提示要迁移到新客户端写法。
# 新写法：先创建一个 OpenAI() 客户端对象，再用 client.chat.completions.create(...) 调用。
client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])

def get_completion(prompt,model=llm_model):
    messages=[{"role": "user", "content": prompt}]
    response=client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0  # temperature=0 表示输出更确定、更少随机性
    )
    # 新版 response 是一个 Pydantic 对象，字段用属性访问（.content），而不是旧版的字典写法 message["content"]
    return response.choices[0].message.content

In [ ]:
# 最简单的调用测试：直接问一个问题（真实网络请求，本环境用的是假 key，这里会在联网请求阶段报认证错误，属预期）
get_completion("What is 1+1?")

In [ ]:
# 一段带有海盗口吻、语气比较激动的客户邮件（示例文本），后面要用 LLM 把它转换成平和、专业的语气
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [ ]:
# 目标风格描述：用于告诉模型要把文本改写成什么语气/风格
style = """American English \
in a calm and respectful tone
"""

In [ ]:
# 用 Python 原生 f-string 手工拼接 prompt（还没用到 LangChain 的 PromptTemplate）
# 三重反引号 ``` 用来给模型划定"需要翻译的文本"边界，避免模型把指令和正文混淆
prompt = f"""Translate the text \
that is delimited by triple backticks
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

In [ ]:
# 调用刚才封装的 get_completion 函数，发起真实请求（此处会因为 .env 里是假 key 而认证失败，属预期行为）
response = get_completion(prompt)

In [ ]:
response

## Chat API : LangChain

上面一节直接用 openai 官方库调用模型；从这里开始改用 LangChain 封装的 `ChatOpenAI`，
好处是后面可以无缝切换到 `PromptTemplate`、`OutputParser`、`Chain` 等 LangChain 生态组件，
不用自己手写字符串拼接和结果解析逻辑。

In [ ]:
# 【版本兼容性修复】原代码是 from langchain_community.chat_models import ChatOpenAI，
# 在当前装的 langchain-community 0.4.2 中已经不再提供 ChatOpenAI（会 ImportError）。
# ChatOpenAI 现在被拆分到独立的 langchain_openai 包中，这是官方推荐的新导入路径。
from langchain_openai import ChatOpenAI

In [ ]:
# 创建一个 LangChain 的 ChatOpenAI 对象，封装了对 OpenAI Chat 模型的调用
# temperature=0.0 表示输出尽量确定、可复现
chat=ChatOpenAI(temperature=0.0,model=llm_model)
chat

In [ ]:
# 定义一个"模板字符串"，{style} 和 {text} 是占位符，之后会用 LangChain 的 PromptTemplate 填充
# 这是 LangChain 核心概念之一：把 prompt 结构和具体内容解耦，方便复用同一模板处理不同输入
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [ ]:
# ChatPromptTemplate.from_template 会把普通字符串模板解析成一个"消息模板"对象
# （内部会把它包装成一条 HumanMessage 模板），后续可以用 .format_messages(...) 填充占位符
from langchain_core.prompts import ChatPromptTemplate
prompt_template=ChatPromptTemplate.from_template(template_string)

In [ ]:
# 查看第一条消息模板底层的 PromptTemplate 对象（可以看到 input_variables、原始模板字符串等信息）
prompt_template.messages[0].prompt

In [ ]:
# 查看模板中识别出的占位符变量名，应该是 ['style', 'text']
prompt_template.messages[0].prompt.input_variables

In [ ]:
# 顾客想要的目标风格：美式英语、语气平和、有礼貌
customer_style = """American English \
in a calm and respectful tone
"""

In [ ]:
# 同样的海盗风客户邮件，重新定义一遍，方便下面用模板填充演示
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [ ]:
# 用 format_messages 把 style/text 占位符替换成真实内容，返回一个消息列表（List[BaseMessage]）
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [ ]:
# customer_messages 是一个 list；第一个元素是 HumanMessage 对象（LangChain 用消息对象而不是纯字符串表示对话）
print(type(customer_messages))
print(type(customer_messages[0]))

In [ ]:
# Call the LLM to translate to the style of the customer message
# 【版本兼容性修复】原代码直接把 chat 对象当函数调用：chat(customer_messages)。
# 这是旧版 LangChain 的用法（ChatModel.__call__）。当前 langchain_openai.ChatOpenAI
# 已经不是 callable 对象了，直接调用会报 "'ChatOpenAI' object is not callable"。
# 新版统一用 Runnable 接口的 .invoke(...) 方法来调用。
customer_response = chat.invoke(customer_messages)

In [ ]:
# invoke() 返回的是一个 AIMessage 对象，真正的文本内容在 .content 属性里
print(customer_response.content)

In [ ]:
# 客服的回复原文：语气比较生硬、不礼貌
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [ ]:
# 这次目标风格反过来：把客服的话改写成"礼貌的英式海盗腔"，演示同一模板可复用于不同方向的转换
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [ ]:
# 复用同一个 prompt_template，只是换了 style 和 text 的值，体现模板可复用的优势
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

In [ ]:
# 同样地，这里也要用 .invoke() 而不是把 chat 当函数直接调用（原理同上一处修复）
service_response = chat.invoke(service_messages)
print(service_response.content)

## Output Parsers

目标：把模型返回的自由文本，解析成结构化的 Python 对象（如 dict），方便后续代码直接使用字段值，
而不是每次都靠人眼去读文本、或者用不可靠的字符串处理去"猜"字段内容。
下面先演示"裸提示 JSON 格式"的局限性，再引入 `StructuredOutputParser` 做更可靠的结构化解析。

In [ ]:
# 这不是一段可执行代码，只是用来展示"我们希望模型最终输出成这种 JSON 结构"的示例
# （字面量 False 在 Python 里可以直接写，不加引号，但作为 markdown/示例展示更直观）
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

In [ ]:
# 一段商品评价文本，接下来要让 LLM 从中抽取结构化信息（是否是礼物、送货天数、价格相关描述）
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

# 用自然语言在 prompt 里描述"要提取哪些字段、每个字段的含义、输出格式是 JSON"
# 这种"裸提示 JSON 格式"的做法不够可靠（模型可能输出格式不规范的 JSON），
# 后面会引入 StructuredOutputParser 来生成更严格的格式说明并做解析
review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [ ]:
# 用上面的 review_template 构建一个 ChatPromptTemplate（这里的 import 前面已经导入过，重复写一遍是原教程的写法，不影响运行）
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

In [ ]:
# 填充模板生成消息，并调用模型（直接问，不用 output parser，先看看模型"裸输出"的效果）
# 【版本兼容性修复】同前面一样，chat(messages) 直接调用已不可用，需改成 chat.invoke(messages)
messages = prompt_template.format_messages(text=customer_review)
chat = ChatOpenAI(temperature=0.0, model=llm_model)
response = chat.invoke(messages)
print(response.content)

In [ ]:
# 这行代码是刻意设计的"反面教材"，运行会报错：
# response.content 是模型返回的纯文本字符串（哪怕看起来像 JSON，本质仍是 str），
# str 类型没有 .get() 方法，所以调用 response.content.get('gift') 会抛 AttributeError。
# 这正是为什么我们需要一个真正的输出解析器（output parser）把字符串解析成 dict，
# 而不是简单地假设模型输出"看起来像 JSON"就能当字典用。
# You will get an error by running this line of code
# because'gift' is not a dictionary
# 'gift' is a string
response.content.get('gift')

In [ ]:
# 【版本兼容性修复】原代码是 from langchain_core.output_parsers import ResponseSchema, StructuredOutputParser。
# 但 ResponseSchema / StructuredOutputParser 属于比较"重"的、依赖 LLMChain 等的旧式输出解析器，
# 在新版拆分中它们被放进了 langchain_classic（原 langchain 主库中偏"经典链路"的部分），
# 而不在轻量级的 langchain_core.output_parsers 里，用原路径会直接 ImportError。
from langchain_classic.output_parsers import ResponseSchema
from langchain_classic.output_parsers import StructuredOutputParser

In [ ]:
# 用 ResponseSchema 逐个定义希望模型输出的字段：字段名 name + 字段含义描述 description
# StructuredOutputParser 会根据这些 schema 自动生成"格式说明"文本，并在解析阶段按 schema 校验/提取字段
gift_schema = ResponseSchema(name="gift",
                             description="Was the item purchased\
                             as a gift for someone else? \
                             Answer True if yes,\
                             False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days",
                                      description="How many days\
                                      did it take for the product\
                                      to arrive? If this \
                                      information is not found,\
                                      output -1.")
price_value_schema = ResponseSchema(name="price_value",
                                    description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list.")

response_schemas = [gift_schema,
                    delivery_days_schema,
                    price_value_schema]

In [ ]:
# 用 response_schemas 列表构造出一个结构化输出解析器
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [ ]:
# get_format_instructions() 会生成一段"教模型如何输出 markdown JSON 代码块"的说明文字，
# 我们把这段说明拼进 prompt 里，让模型按照约定格式回复，从而提高解析成功率
format_instructions = output_parser.get_format_instructions()

In [ ]:
print(format_instructions)

In [ ]:
# 新模板在原来的基础上多了 {format_instructions} 占位符，用来插入上面生成的格式说明
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

# 同时传入 text 和 format_instructions 两个变量，填充出最终的消息内容
messages = prompt.format_messages(text=customer_review,
                                format_instructions=format_instructions)

In [ ]:
# 打印最终发给模型的完整 prompt 文本，可以看到末尾多了格式说明部分
print(messages[0].content)

In [ ]:
# 【版本兼容性修复】同前面一样，chat(messages) 直接调用已不可用，需改成 chat.invoke(messages)
response = chat.invoke(messages)

In [ ]:
# 这一次模型输出的应该是被 ```json ... ``` 包裹的、符合约定 schema 的文本
print(response.content)

In [ ]:
# output_parser.parse() 会去掉 markdown 代码块标记，把里面的 JSON 文本解析成真正的 Python dict
# （已在本地用构造出的假数据验证过：parse() 能正确处理 ```json ... ``` 包裹的字符串并返回 dict）
output_dict = output_parser.parse(response.content)

In [ ]:
output_dict

In [ ]:
# 这里应该输出 <class 'dict'>，和前面 response.content.get('gift') 报错时的 str 类型形成对比
type(output_dict)

In [ ]:
# 现在 output_dict 真的是字典了，.get() 可以正常使用
output_dict.get('delivery_days')